[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/solutions/65_adaln_zero_solution.ipynb)

# 🟡 Solution: AdaLN-Zero

Reference solution for Adaptive Layer Norm Zero (AdaLN-Zero) used in DiT and similar diffusion transformers.

A conditioning vector modulates the layer norm output via learned scale, shift, and gate parameters:

$$\text{out} = \alpha_1 \cdot (\gamma_1 \cdot \text{LN}(x) + \beta_1)$$

All six parameters $(\gamma_1, \beta_1, \alpha_1, \gamma_2, \beta_2, \alpha_2)$ are predicted from the conditioning vector. At initialization $W_{\text{ada}} = 0$, so $\alpha_1 = 0$ and the block contributes nothing (identity residual).

In [ ]:
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass

In [ ]:
import torch

In [ ]:
# ✅ SOLUTION

def adaln_zero(x, cond, W_ada, b_ada):
    B, N, D = x.shape
    params = cond @ W_ada + b_ada
    chunks = params.chunk(6, dim=-1)
    gamma1, beta1, alpha1 = chunks[0], chunks[1], chunks[2]
    mean = x.mean(dim=-1, keepdim=True)
    var = x.var(dim=-1, keepdim=True, unbiased=False)
    x_norm = (x - mean) / (var + 1e-6).sqrt()
    out = alpha1.unsqueeze(1) * (gamma1.unsqueeze(1) * x_norm + beta1.unsqueeze(1))
    return out

In [ ]:
torch.manual_seed(0)

B, N, D = 4, 16, 32
C = 16  # conditioning dim

x    = torch.randn(B, N, D)
cond = torch.randn(B, C)

# Zero W_ada => all params = b_ada; if b_ada is also zero => alpha1=0 => output is zero
W_ada = torch.zeros(C, 6 * D)
b_ada = torch.zeros(6 * D)
out_zero = adaln_zero(x, cond, W_ada, b_ada)
print(f"Zero W_ada, zero b_ada => max abs output: {out_zero.abs().max().item():.6f}  (expected 0.0)")

# Non-zero W_ada => output should be non-zero
W_ada_rand = torch.randn(C, 6 * D) * 0.1
b_ada_rand = torch.randn(6 * D) * 0.1
out_rand = adaln_zero(x, cond, W_ada_rand, b_ada_rand)
print(f"Random W_ada          => output shape: {out_rand.shape}, mean abs: {out_rand.abs().mean().item():.4f}")

In [ ]:
from torch_judge import check
check("adaln_zero")